<a href="https://colab.research.google.com/github/estefania-apaza/state-capacity-protest-peru/blob/main/Avance_de_base_de_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Avance de la Base de Datos
Propuesta de Investigación
- Curso: Estadística para el Análisis Político 2
- Nombres: Estefanía Apaza (20230487) y Diego Luyo (20230934)


Insumos

[Libro de Códigos](https://docs.google.com/spreadsheets/d/1U5zJE58q_83H3Lc0-il9jx8QrDb9EX78/edit?gid=266433641#gid=266433641)

[Base de Eventos de Protesta](https://github.com/estefania-apaza/state-capacity-protest-peru/blob/main/Base%20de%20Eventos%20de%20Protesta%20del%20Peru%CC%81_1980_2025.csv) (Aragón, et al.) - Necesario subirla a Google Colab para correr el código

### Avance de la variable Y


**1. Limpieza y filtrado de la base de datos**

In [65]:
import pandas as pd

# Cargado de la base de la Escuela de Gobierno y Políticas Públicas PUCP
file_path = "Base de Eventos de Protesta del Perú_1980_2025.csv"
df_egpp = pd.read_csv(file_path, sep=';', encoding='latin-1', on_bad_lines='skip')

# Limpieza de nombres de columnas
df_egpp.columns = df_egpp.columns.str.strip().str.lower()
df_egpp.columns = df_egpp.columns.str.replace('ñ', 'n').str.replace('á', 'a').str.replace('é', 'e').str.replace('í', 'i').str.replace('ó', 'o').str.replace('ú', 'u')

# Selección preliminar de columnas
columnas_analisis = [
    'id', 'ano', 'mes_id', 'presidente_id',
    'region_id', 'provincia_id', 'distrito_id',
    'sector_id_1', 'actor_1_id', 'sector_id_2', 'actor_2_id',
    'accion_1_id', 'accion_2_id', 'accion_3_id', 'accion_4_id',
    'duracion_horas', 'numero_participantes',
    'numero_heridos', 'numero_muertos', 'numero_detenidos',
    'adversario_id', 'institucion_id',
    'reclamo_id', 'sub_reclamo_id']

# Creación de la base preliminar
df_preliminar = df_egpp[[c for c in columnas_analisis if c in df_egpp.columns]].copy()

df_preliminar.columns
df_preliminar.head(5)

,id,ano,mes_id,presidente_id,region_id,provincia_id,distrito_id,sector_id_1,actor_1_id,sector_id_2,...,accion_4_id,duracion_horas,numero_participantes,numero_heridos,numero_muertos,numero_detenidos,adversario_id,institucion_id,reclamo_id,sub_reclamo_id
0,1,1980,1,1,15,1501,150101,18,102,NaN,...,NaN,24.0,NaN,NaN,NaN,NaN,9,909.0,1,105
1,2,1980,1,1,15,1501,150101,13,104,NaN,...,NaN,24.0,2500.0,NaN,NaN,NaN,7,712.0,1,101
2,3,1980,1,1,15,1501,150108,103,206,NaN,...,NaN,24.0,NaN,15.0,NaN,NaN,1,105.0,4,407
3,4,1980,1,1,15,1501,150101,26,127,NaN,...,NaN,24.0,NaN,NaN,NaN,NaN,9,909.0,1,104
4,5,1980,1,1,8,801,80101,21,119,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,5,508.0,1,109


**2. Definición del umbral de la violencia para la variable dependiente: "Protesta violenta"**

Criterios

*   Alguna de las acciones implica violencia según el Libro de Códigos
*   Existe alguna víctima física



In [66]:
def variable_violencia(row):
    # Códigos de Acción seleccionados por implicar violencia según el Libro de Códigos (107-114, 119, 123)
    codigo_violencia = [107, 108, 109, 110, 111, 112, 113, 114, 119, 123]

    # Extraemos los IDs mencionados de las columnas de Acción (1-4)
    acciones = [row.get('accion_1_id'), row.get('accion_2_id'),
                row.get('accion_3_id'), row.get('accion_4_id')]

    # Criterio 1 - Acción implica violencia
    es_accion_violenta = any(acc in codigo_violencia for acc in acciones)

    # Criterio 2 - Existen víctimas físicas
    muertos = pd.to_numeric(row.get('numero_muertos', 0), errors='coerce') or 0
    heridos = pd.to_numeric(row.get('numero_heridos', 0), errors='coerce') or 0
    tiene_victimas = (muertos > 0 or heridos > 0)

    if es_accion_violenta or tiene_victimas:
        return 1
    return 0

# Creamos la variable dependiente con la función
df_preliminar['violencia_y'] = df_preliminar.apply(variable_violencia, axis=1)

print(df_preliminar['violencia_y'].value_counts())

violencia_y
0    21144
1     3882
Name: count, dtype: int64


In [67]:
columnas_final = [
    'id', 'ano', 'mes_id', 'presidente_id',
    'region_id', 'provincia_id', 'distrito_id',
    'sector_id_1', 'actor_1_id', 'sector_id_2', 'actor_2_id',
    'duracion_horas', 'numero_participantes','numero_detenidos',
    'adversario_id', 'institucion_id',
    'reclamo_id', 'sub_reclamo_id', 'violencia_y']

# Creación de la base final para la variable Y
df_final = df_preliminar[[c for c in columnas_final if c in df_preliminar.columns]].copy()

df_final.head(5)

,id,ano,mes_id,presidente_id,region_id,provincia_id,distrito_id,sector_id_1,actor_1_id,sector_id_2,actor_2_id,duracion_horas,numero_participantes,numero_detenidos,adversario_id,institucion_id,reclamo_id,sub_reclamo_id,violencia_y
0,1,1980,1,1,15,1501,150101,18,102,NaN,NaN,24.0,NaN,NaN,9,909.0,1,105,0
1,2,1980,1,1,15,1501,150101,13,104,NaN,NaN,24.0,2500.0,NaN,7,712.0,1,101,0
2,3,1980,1,1,15,1501,150108,103,206,NaN,NaN,24.0,NaN,NaN,1,105.0,4,407,1
3,4,1980,1,1,15,1501,150101,26,127,NaN,NaN,24.0,NaN,NaN,9,909.0,1,104,0
4,5,1980,1,1,8,801,80101,21,119,NaN,NaN,NaN,NaN,NaN,5,508.0,1,109,0


**3. Revisión de la base**

In [68]:
print("Descripción del avance de la Base de Datos")
print(f"Columnas: {len(df_preliminar.columns)}")
print(f"Observaciones: {len(df_final)}")
print("\nConteo de la Variable Dependiente (Protesta violenta):")
print(df_preliminar['violencia_y'].value_counts())

Descripción del avance de la Base de Datos
Columnas: 25
Observaciones: 25026

Conteo de la Variable Dependiente (Protesta violenta):
violencia_y
0    21144
1     3882
Name: count, dtype: int64


**Referencias**

Aragón, Jorge, Moisés Arce, Renzo Aurazo y Omar Coronel. 2025. Base de Eventos de Protestas del Perú, Versión Agosto 2025. Lima: Pontificia Universidad Católica del Perú-PUCP

Aragón, Jorge, Moisés Arce, Renzo Aurazo y Omar Coronel. 2025. Base de Eventos de Protestas del Perú: Libro de Códigos, Versión Agosto 2025. Lima: Pontificia Universidad Católica del Perú-PUCP

### Limpieza de duplicados y minimización del sesgo de clases

In [69]:
# Ordenamos la base: los 1 (violencia) primero, los 0 (no violento) después
df_preliminar = df_preliminar.sort_values(by='violencia_y', ascending=False)

# Definimos nuestra llave de identidad
llave_duplicados = ['ano', 'mes_id', 'region_id', 'provincia_id', 'distrito_id', 'actor_1_id', 'reclamo_id']

# Borramos duplicados manteniendo la primera observación
antes = len(df_preliminar)
df_preliminar = df_preliminar.drop_duplicates(subset=llave_duplicados, keep='first')
despues = len(df_preliminar)

print(f"Filas eliminadas: {antes - despues}")
print(f"Nueva distribución de Y:\n{df_preliminar['violencia_y'].value_counts()}")

Filas eliminadas: 7311
Nueva distribución de Y:
violencia_y
0    14668
1     3047
Name: count, dtype: int64


In [73]:
# Acciones simbólicas para filtrar y eliminar
# 127: Olla común, 122: Lavados, 115: Mitín, 129: Protesta simbólica, 125: Cacerolazo
acciones_simbolicas = [127, 122, 115, 129, 125]

def es_evento_prescindible(row):
    acciones_evento = [row['accion_1_id'], row['accion_2_id'],
                       row['accion_3_id'], row['accion_4_id']]
    acciones_reales = [int(a) for a in acciones_evento if pd.notnull(a)]
    # Si todas son simbólicas, el evento es prescindible
    return all(a in acciones_simbolicas for a in acciones_reales)

# Aplicamos el filtro para quedarnos con las no prescindible
df_preliminar = df_preliminar[~df_preliminar.apply(es_evento_prescindible, axis=1)]

print(f"Nueva distribución tras quitar acciones simbólicas puras:\n{df_preliminar['violencia_y'].value_counts()}")

Nueva distribución tras quitar acciones simbólicas puras:
violencia_y
0    14540
1     3047
Name: count, dtype: int64


### Avance de la variable X - Autoridad estatal

**1. Elaboración de la función para la variable independiente 1: "Autoridad Estatal"**

* Proxy de Sullivan: Reportes de crímenes
* Propuesta: Acceso a la justicia del Módulo de Gobernabilidad de la ENAHO

1. "En los últimos 12 meses, ¿ha tenido algún desacuerdo, conflicto o problema...?"

2. "Por este desacuerdo o conflicto acudió ud. a:"

* Uso del Estado: 7-12
* No uso del Estado: 1-6


In [71]:
# Cargado de el módulo de Gobernabilidad de la ENAHO 2024 como inicio
file_path = "Enaho01B-2024-2.csv"
enaho2024 = pd.read_csv(file_path, sep=',', encoding='latin-1', on_bad_lines='skip', low_memory=False)
enaho2024.columns = enaho2024.columns.str.strip().str.upper()

if 'AÃ‘O' in enaho2024.columns:
    enaho2024.rename(columns={'AÃ‘O': 'ANIO'}, inplace=True)
elif 'AÑO' in enaho2024.columns:
    enaho2024.rename(columns={'AÑO': 'ANIO'}, inplace=True)

FileNotFoundError: [Errno 2] No such file or directory: 'Enaho01B-2024-2.csv'

In [ ]:
def definir_autoridad(df, anio):
    # Definimos las columnas
    cols_conflictos = [f'P24_{i}' for i in range(1, 12)]
    cols_estado = [f'P26_{i}' for i in [3, 4, 5, 7, 8, 9, 10, 12]]

    # Convertimos a números
    for c in cols_conflictos + cols_estado:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    # Identificar quién tuvo al menos un conflicto (1 = Sí)
    df_proceso = df.copy()
    df_proceso['tuvo_conflicto'] = (df_proceso[cols_conflictos] == 1).any(axis=1).astype(int)

    # Filtrar por víctimas
    df_victimas = df_proceso[df_proceso['tuvo_conflicto'] == 1].copy()

    # Variable de quiénes acudieron al Estado
    df_victimas['acudio_estado'] = (df_victimas[cols_estado] == 1).any(axis=1).astype(int)

    # Convertir Ubigeo a string con ceros
    df_victimas['region'] = df_victimas['UBIGEO'].astype(str).str.zfill(6).str[:2]

    # Calcular porcentaje por región
    resultado = df_victimas.groupby('region')['acudio_estado'].mean().reset_index()
    resultado['ano'] = anio
    resultado.rename(columns={'acudio_estado': 'autoridad_x1'}, inplace=True)
    resultado['autoridad_x1'] = resultado['autoridad_x1']*100

    return resultado

# Ejecución de prueba
df_x_2024 = definir_autoridad(enaho2024, 2024)
print(df_x_2024.head())

Aplicación en las bases de datos ENAHO 2018-2023 (2024 realizado previamente).

In [ ]:
nombres_bases = []
dataframes_cargados = []

for i in range(18, 25):
  bases = f'Enaho01B-20{i}-2.csv'
  nombres_bases.append(bases)

for base in nombres_bases:
  file_path = base
  base = pd.read_csv(file_path, sep=',', encoding='latin-1', on_bad_lines='skip', low_memory=False)
  base.columns = base.columns.str.strip().str.upper()
  if 'AÃ‘O' in base.columns:
    base.rename(columns={'AÃ‘O': 'ANIO'}, inplace=True)
  elif 'AÑO' in base.columns:
    base.rename(columns={'AÑO': 'ANIO'}, inplace=True)
  dataframes_cargados.append(base)

enaho2018, enaho2019, enaho2020, enaho2021, enaho2022, enaho2023, enaho2024 = dataframes_cargados

In [ ]:
resultados_anuales = []
anios = [2018, 2019, 2020, 2021, 2022, 2023, 2024]

for enaho, anio in zip(dataframes_cargados, anios):
    print(f"Procesando año: {anio}")

    # Aplicamos función
    df_resultado = definir_autoridad(enaho, anio)

    resultados_anuales.append(df_resultado)

df_final_x1 = pd.concat(resultados_anuales, ignore_index=True)

In [ ]:
df_cruzado = df_final_x1.groupby(['region', 'ano'])['autoridad_x1'].mean().reset_index()
print(df_cruzado)

**Unir ambas bases para su análisis**

In [ ]:
#df_final['region_id'] = df_final['region_id'].astype(str).str.zfill(2)
#df_cruzado['region_id'] = df_cruzado['region_id'].astype(str).str.zfill(2)

#df_paper = pd.merge(
#    df_final,
#    df_cruzado,
#    on=['region_id', 'ano'],
#    how='left'
#)

#print("Filas totales:", len(df_paper))
#print("Valores de autoridad_x1 encontrados:\n", df_paper['autoridad_x1'].notnull().value_counts())

**Referencias**

Instituto Nacional de Estadística e Informática. (2024). Encuesta Nacional de Hogares (ENAHO) [Base de datos]. https://proyectos.inei.gob.pe/microdatos/

Instituto Nacional de Estadística e Informática. (2023). Encuesta Nacional de Hogares (ENAHO) [Base de datos]. https://proyectos.inei.gob.pe/microdatos/

Instituto Nacional de Estadística e Informática. (2022). Encuesta Nacional de Hogares (ENAHO) [Base de datos]. https://proyectos.inei.gob.pe/microdatos/

Instituto Nacional de Estadística e Informática. (2021). Encuesta Nacional de Hogares (ENAHO) [Base de datos]. https://proyectos.inei.gob.pe/microdatos/

Instituto Nacional de Estadística e Informática. (2020). Encuesta Nacional de Hogares (ENAHO) [Base de datos]. https://proyectos.inei.gob.pe/microdatos/

Instituto Nacional de Estadística e Informática. (2019). Encuesta Nacional de Hogares (ENAHO) [Base de datos]. https://proyectos.inei.gob.pe/microdatos/

Instituto Nacional de Estadística e Informática. (2018). Encuesta Nacional de Hogares (ENAHO) [Base de datos]. https://proyectos.inei.gob.pe/microdatos/

### Avance de las variables de control

1. **Protesta masiva**

In [ ]:
# Evaluamos corte para protesta masiva

df_final['numero_participantes'].quantile([0.25, 0.5, 0.75, 0.9, 0.95])

In [ ]:
# Seleccionamos el umbral

umbral_elegido = 3500

# Elaboramos la variable

df_final['protesta_masiva'] = (df_final['numero_participantes'] >= umbral_elegido).astype(int)

print(df_final['protesta_masiva'].value_counts())

2. **Número de eventos**

In [ ]:
# Agrupamos para contar cuántas protestas hubo por provincia cada mes
conteo_mensual = df_final.groupby(['provincia_id', 'ano', 'mes_id']).size().reset_index(name='n_eventos_mes')

# Ordenamos para que el desplazamiento sea cronológico por provincia
conteo_mensual = conteo_mensual.sort_values(['provincia_id', 'ano', 'mes_id'])

# Creamos la sección del valor del mes anterior se pone en la fila del mes actual
conteo_mensual['numero_eventos_previos'] = conteo_mensual.groupby('provincia_id')['n_eventos_mes'].shift(1).fillna(0)

df_final = pd.merge(
    df_final,
    conteo_mensual[['provincia_id', 'ano', 'mes_id', 'numero_eventos_previos']],
    on=['provincia_id', 'ano', 'mes_id'],
    how='left'
)

df_final['numero_eventos_previos'] = df_final['numero_eventos_previos'].fillna(0)

print("--- RESUMEN DE LA BASE ACTUALIZADA ---")
print(f"Total de filas: {len(df_final)}")
print(f"Protestas masivas (>= 3500): {df_final['protesta_masiva'].sum()}")
print("\nMuestra de inercia (eventos el mes pasado):")
print(df_final[['provincia_id', 'ano', 'mes_id', 'protesta_masiva', 'numero_eventos_previos']].tail())




---


In [ ]:
# Descarga del avance de la base de datos
from google.colab import files

df_final.to_csv('base_protestas_procesada.csv', index=False, encoding='utf-8-sig')
#files.download('base_protestas_procesada.csv')